In [17]:
import argparse
import os
import pingouin
import numpy as np
import os.path as op
import pandas as pd
from nilearn import surface
from braincoder.optimize import ResidualFitter
from braincoder.models import GaussianPRF
from braincoder.utils import get_rsq
import numpy as np
from stress_risk.utils import Subject
from braincoder.models import GaussianPRF
from braincoder.optimize import ParameterFitter

stimulus_range = np.linspace(0, 6, 1000)
# stimulus_range = np.log(np.arange(400))
roi = 'NPC_R'#'wang15_ips'
space = 'T1w'
smoothed = False
retroicor = False
pca_confounds = False
denoise = True

In [4]:
subject = '01'
session = 1
bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'

target_dir = op.join(bids_folder, 'derivatives', 'decoded_pdfs.volume.svoxels.ses1-en_ses2de')
target_dir = op.join(target_dir, f'sub-{subject}', 'func')
if not op.exists(target_dir):
    os.makedirs(target_dir)


In [5]:
sub = Subject(subject, bids_folder)

paradigm1 = sub.get_behavior(sessions=1, drop_no_responses=False)
paradigm1['log(n1)'] = np.log(paradigm1['n1'])
paradigm1 = paradigm1.droplevel(['subject', 'session'])
paradigm1 = paradigm1['log(n1)']

paradigm2 = sub.get_behavior(sessions=2, drop_no_responses=False)
paradigm2['log(n1)'] = np.log(paradigm2['n1'])
paradigm2 = paradigm2.droplevel(['subject', 'session'])
paradigm2 = paradigm2['log(n1)']

In [48]:
data1 = sub.get_single_trial_volume(1, roi=roi, smoothed=smoothed, pca_confounds=pca_confounds, denoise=denoise, retroicor=retroicor).astype(np.float32)
data1.index = paradigm1.index

data2 = sub.get_single_trial_volume(2, roi=roi, smoothed=smoothed, pca_confounds=pca_confounds, denoise=denoise, retroicor=retroicor).astype(np.float32)
data2.index = paradigm2.index

/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/image/image.py:756: FutureWarning: Image data has type int64, which may cause incompatibilities with other tools. This will error in NiBabel 5.0. This warning can be silenced by passing the dtype argument to Nifti1Image().
  return klass(data, affine, header=header)
/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/image/image.py:756: FutureWarning: Image data has type int64, which may cause incompatibilities with other tools. This will error in NiBabel 5.0. This warning can be silenced by passing the dtype argument to Nifti1Image().
  return klass(data, affine, header=header)


In [49]:
test_data, test_paradigm = data1.copy(), paradigm1.copy()
train_data, train_paradigm = data2.copy(), paradigm2.copy()


In [52]:

from nilearn import image
from nilearn.maskers import NiftiMasker

fn = '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/encoding_model.cv.denoise/sub-01/ses-1/func/sub-01_ses-1_desc-cvr2.optim_space-T1w_pars.nii.gz'
im_cvr2 = image.load_img(fn)

mask = sub.get_volume_mask(roi=roi, session=1, epi_space=True) # anat from session1
masker = NiftiMasker(mask_img=mask)

cv_r2 = pd.DataFrame(masker.fit_transform(im_cvr2))
r2_mask = cv_r2 > 0.0
r2_mask = r2_mask.to_numpy().T


/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/image/image.py:756: FutureWarning: Image data has type int64, which may cause incompatibilities with other tools. This will error in NiBabel 5.0. This warning can be silenced by passing the dtype argument to Nifti1Image().
  return klass(data, affine, header=header)


In [54]:
train_data = train_data.loc[:, r2_mask]
test_data = test_data.loc[:, r2_mask]

In [56]:
pars = sub.get_prf_parameters_volume(session=1, cross_validated=False,
        denoise=denoise, retroicor=retroicor,
        smoothed=smoothed, pca_confounds=pca_confounds,
        roi=roi)

model = GaussianPRF(parameters=pars)

model.apply_mask(r2_mask)
model.init_pseudoWWT(stimulus_range, model.parameters)
residfit = ResidualFitter(model, train_data,
                            train_paradigm.astype(np.float32))

omega, dof = residfit.fit(init_sigma2=10.0,
        method='t',
        max_n_iterations=10000)

print('DOF', dof)

/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nilearn/image/image.py:756: FutureWarning: Image data has type int64, which may cause incompatibilities with other tools. This will error in NiBabel 5.0. This warning can be silenced by passing the dtype argument to Nifti1Image().
  return klass(data, affine, header=header)
2023-05-17 13:20:20.722825: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2023-05-17 13:20:20.724690: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2023-05-17 13:20:20.988655: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2023-05-17 13:20:20.

Metal device set to: Apple M1 Pro

systemMemory: 16.00 GB
maxCacheSize: 5.33 GB



2023-05-17 13:20:21.937099: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


(120, 47)
init_tau: 0.7309564352035522, 2.0136873722076416
USING A PSEUDO-WWT!
WWT max: 5961.681640625


  0%|          | 0/10000 [00:00<?, ?it/s]2023-05-17 13:20:23.247947: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2023-05-17 13:20:23.521267: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2023-05-17 13:20:23.744834: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2023-05-17 13:20:24.489571: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2023-05-17 13:20:25.238233: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2023-05-17 13:20:26.117931: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
fit stat: 9004.3418 (best: 90

DOF 29.338581


In [58]:
r2_mask.sum()

47

In [59]:
bins = stimulus_range.astype(np.float32)

pdf = model.get_stimulus_pdf(test_data, bins,
        model.parameters,
        omega=omega,
        dof=dof)


print(pdf)
E = (pdf * pdf.columns).sum(1) / pdf.sum(1)

print(pd.concat((E, test_paradigm), axis=1))
print(pingouin.corr(E, test_paradigm))


2023-05-17 13:26:47.983861: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


stimulus          0.000000      0.006006      0.012012      0.018018  \
run trial_nr                                                           
1   1         5.429441e-07  5.600978e-07  5.777582e-07  5.959391e-07   
    2         2.384375e-06  2.439132e-06  2.494955e-06  2.551862e-06   
    3         4.996526e-07  5.118217e-07  5.242510e-07  5.369495e-07   
    4         3.918022e-05  4.021844e-05  4.128071e-05  4.236715e-05   
    5         2.899201e-03  2.915571e-03  2.931765e-03  2.947734e-03   
...                    ...           ...           ...           ...   
6   116       4.614459e-06  4.700914e-06  4.788806e-06  4.878118e-06   
    117       1.949006e-04  1.993521e-04  2.038881e-04  2.085036e-04   
    118       1.623737e-03  1.644332e-03  1.665010e-03  1.685807e-03   
    119       1.321354e-06  1.346521e-06  1.372115e-06  1.398121e-06   
    120       4.024726e-06  4.118033e-06  4.213278e-06  4.310496e-06   

stimulus          0.024024      0.030030      0.036036      0.0